## Faster inference - vLLM

vLLM is library optimized for inference - hence it provides better GPU utilization and much faster inference than base transformers library. vLLM offers two usage options:
- [vLLM server](https://docs.vllm.ai/en/latest/serving/openai_compatible_server/): Creates an inference server (compatible with OpenAI library), that can be used for API access;
- **Local inference**: Creates a LLM class that can be used inside the running script. We will use this option today.

Let us first load the model. We will load the model in 16-bit precision. If we had memory problems, we could load the model in 4-bit quantization by adding `quantization="bitsandbytes"`. An important additional parameter for vLLM is  `gpu_memory_utilization`. It tells the vLLM engine, what percentage of GPU memory it should reserve for model weights and KV-cache. By default it is set to `0.9`. The rule of thumb for 16-bit percision is to reserve the amount of vRAM equal to 2-times the size of the model plus additional 20-50 % for KV-cache (the second value depends on the maximum sequence length). Since we are loading 12B model, we need approximatelly 24 GB for parameters and approximatelly 10 GB for the KV cache at full sequence length (in vLLM we can also use shorter context windows by setting `max_model_len` parameter). Since we are working on a 128 GB DGX Spark, we will set `gpu_memory_utilization=0.4`

In [ ]:
from vllm import LLM, SamplingParams
import torch

model = LLM(
    "/models/GaMS3-12B-Instruct",
    dtype=torch.bfloat16,
    gpu_memory_utilization=0.4
)

## Inference

### Prompts

Let's first define our list of prompts. We will use the set of prompts that can appear when working with a local assistant set up by a company. Some prompts are intentionally project or company specific (projekt Triglav, varnostni protokoli). We will see their purpose later when testing RAG.

In [ ]:
prompts = [
    "Napiši kratek Python primer za uporabo knjižnice pandas.",
    "Kakšen je postopek za delo od doma za zaposlene z manj kot letom dni delovne dobe?",
    "Katere korake moram izvesti za postavitev razvojnega okolja pri projektu Triglav?",
    "Pripravi osnutek objave za LinkedIn, v kateri podjetje napoveduje novo sodelovanje na področju umetne inteligence.",
    "Kakšni so varnostni protokoli glede shranjevanja API ključev strank v našem podjetju?"
]

### Inference function

vLLM uses same chat format for inference as transformers. There are two ways of using vLLM for response generation:
- `generate`: used for standard text-completion task (pretrained models). If we would want to use a chat model in this format, we would have to pretokenize the prompt using chat template;
- `chat`: better suited for chat model. Using this method, we can send the conversation directly to the model and its tokenizer will automatically apply chat template (the same way as this is handled in transformers while using `pipeline`). Since we are using chat model, we will use this option today.

**Sampling params**: vLLM uses a special class for generation parameters such as temperature, top_p, top_k, etc. It also supports guided decoding format (enforcing JSON schema or answers given regular expression - we will take a closer look at guided decoding later).

In [ ]:
def prompt_to_conversation(prompt):
    messages = [
      {"role": "user", "content": prompt}
    ]

    return messages

def vllm_generate(model, conversations):
    sampling_params = SamplingParams(
        temperature=0.6,
        top_p=0.9,
        top_k=64,
        max_tokens=8192
    )

    responses = model.chat(conversations, sampling_params)
    predicted_texts = []
    for response in responses:
        prediction = response.outputs[0].text
        predicted_texts.append(prediction)

    return predicted_texts

### Single example inference

With vLLM we can do both single example and batch inference. For single example, we simply need to put our conversation as an input.

In [ ]:
# We will use the first prompt from the list as our toy example
prompt = prompts[0]
conversation = prompt_to_conversation(prompt)

predicted_text = vllm_generate(model, conversation)

print(50*"-")
print("Input text:")
print(prompt)
print()
print()
print("Model's response:")
print(predicted_text[0])
print(50*"-")

### Batch inference

Batch inference is very simple with vLLM. All we need to do is to provide the list of conversations - be careful, each conversation is a list itself, so we need to provide the list of lists.

The main advantage of using batch inference is faster throughput - vLLM process all batch requests in parallel!

In [ ]:
conversations = [prompt_to_conversation(prompt) for prompt in prompts]

predicted_texts = vllm_generate(model, conversations)

for prompt, predicted_text in zip(prompts, predicted_texts):
    print(50*"-")
    print("Input text:")
    print(prompt)
    print()
    print()
    print("Model's response:")
    print(predicted_text)
    print(50*"-")

## Structured Outputs (Guided Decoding)

Standard LLM generation is probabilistic, meaning that even if you ask for a **JSON** format, the model might occasionally add conversational filler (e.g., "Sure, here is your JSON:") or skip a closing bracket. Moreover, for longer and more complex tasks it my confuse some fields or XML tags or completely avoid them. **This can happen to large commercial models as well**. This makes it difficult to use LLMs in automated production pipelines. The solution for such use cases is **guided decoding**.

Guided Decoding forces the model to follow a specific grammar or schema. It achieves this by masking tokens. At every step of the generation, the engine filters out tokens that would violate the predefined structure (e.g., if a JSON key is expected, it won't allow the model to generate a number or a random word). Below is a visualisation of token masking from the [lm-format-enforcer repository](https://github.com/noamgat/lm-format-enforcer?tab=readme-ov-file)

![An example of token masking from the lm-format-enforcer backend.](https://raw.githubusercontent.com/noamgat/lm-format-enforcer/main/docs/Trees.drawio.svg?sanitize=true)

When to use it?
1. **Data Extraction:** Converting unstructured text into structured database entries.
2. **Classification:** Forcing the model to output *only* class label without extra text.
3. **API Integration:** Ensuring the LLM response can be directly passed to another function.

vLLM offers different guided decoding backends, such as **outlines**, **lm-format-enforcer** and **xgrammar**. The default backend setting in vLLM is `auto`. In this setting, vLLM automatically selects the most appropriate backend based on the request. For more options, see their [documentation](https://docs.vllm.ai/en/v0.8.2/features/structured_outputs.html)

### JSON Schema parsing

We will start by extracting information from an e-mail in JSON format.

In [ ]:
instruction = "Iz spodnjega e-poštnega sporočila izvleči ime stranke, telefonsko številko in omenjeni izdelek v JSON formatu."

email_input = """\
Zadeva: Povpraševanje glede zaloge in tehnična pomoč

Spoštovani,

pišem vam, ker me zanima, ali imate na zalogi vaš novi model "Pametni krmilnik vlage AirMaster Pro", 
ki sem ga videl v vašem zadnjem katalogu. Potreboval bi vsaj dve enoti za našo testno komoro.

Poleg tega imam nekaj vprašanj glede garancije, zato bi vas prosil, če me lahko nekdo iz 
tehnične službe pokliče na telefonsko številko 040 555 123, najbolje nekje med 8. in 10. uro zjutraj.

Že vnaprej se vam zahvaljujem za odgovor.

Lep pozdrav,
Andrej Kovač
Direktor logistike, LogiSlo d.o.o.\
"""

full_prompt = f"{instruction}\n\n{email_input}"

#### No guided decoding

Let's first look at the model's response when no guided decoding is used.

In [ ]:
conversation = prompt_to_conversation(full_prompt)

predicted_text = vllm_generate(model, conversation)

print(50*"-")
print("Input text:")
print(full_prompt)
print()
print()
print("Model's response:")
print(predicted_text[0])
print(50*"-")

#### Guided decoding

Let's repeat the same prompt, but this time with guided decoding. We will define a JSON schema we want the model to generate. Then we will adapt our generation function by adding the schema to the sampling parameters.

In [ ]:
from vllm.sampling_params import GuidedDecodingParams

json_schema = {
    "type": "object",
    "properties": {
        "ime": {
            "type": "string",
            "description": "Ime in priimek osebe, ki je poslala e-pošto."
        },
        "telefon": {
            "type": "string", 
            "description": "Telefonska številka stranke v formatu nnn nnn nnn."
        },
        "izdelek": {
            "type": "string",
            "description": "Polno ime izdelka, ki ga stranka omenja ali kupuje."
        }
    },
    "required": ["ime", "telefon", "izdelek"],
    "additionalProperties": False
}

def vllm_generate_json(model, conversations):
    # We create a separate object for guided decoding parameters
    guided_decoding_params = GuidedDecodingParams(json=json_schema)
    
    sampling_params = SamplingParams(
        temperature=0.6,
        top_p=0.9,
        top_k=64,
        max_tokens=8192,
        guided_decoding=guided_decoding_params # We add the guided decoding to the sampling parameters
    )

    responses = model.chat(conversations, sampling_params)
    predicted_texts = []
    for response in responses:
        prediction = response.outputs[0].text
        predicted_texts.append(prediction)

    return predicted_texts

In [ ]:
conversation = prompt_to_conversation(full_prompt)

predicted_text = vllm_generate_json(model, conversation)

print(50*"-")
print("Input text:")
print(full_prompt)
print()
print()
print("Model's response:")
print(predicted_text[0])
print(50*"-")

### Classification problem

Next we will look at the classification problem, where we need the model to output only the predicted label.

In [ ]:
instruction = "Klasificiraj naslednji dnevniški zapis (log entry) v eno izmed kategorij: [INFO, WARNING, ERROR, CRITICAL]."

log_entry = """\
2026-04-13 11:42:01,847 - InternalService - CONNECTION_FAILURE - [DB_NODE_04] 
SocketTimeoutException: Connection timed out (Read failed). 
Retrying in 500ms... Attempt 3 of 5. 
Potential resource exhaustion on the database cluster detected.\
"""

full_prompt = f"{instruction}\n\n{log_entry}"

#### No guided decoding

Let's again first look at the model's response when no guided decoding is used.

In [ ]:
conversation = prompt_to_conversation(full_prompt)

predicted_text = vllm_generate(model, conversation)

print(50*"-")
print("Input text:")
print(full_prompt)
print()
print()
print("Model's response:")
print(predicted_text[0])
print(50*"-")

#### Guided decoding

Let's repeat the same prompt, but this time with guided decoding. This time we won't use JSON schema, but will instead use `choice` option. This option lets us define the list of strings that are accepted as answer.

In [ ]:
def vllm_generate_choice(model, conversations):
    # We create a separate object for guided decoding parameters
    guided_decoding_params = GuidedDecodingParams(
        choice=["INFO", "WARNING", "ERROR", "CRITICAL"]
    )
    
    sampling_params = SamplingParams(
        temperature=0.6,
        top_p=0.9,
        top_k=64,
        max_tokens=8192,
        guided_decoding=guided_decoding_params # We add the guided decoding to the sampling parameters
    )

    responses = model.chat(conversations, sampling_params)
    predicted_texts = []
    for response in responses:
        prediction = response.outputs[0].text
        predicted_texts.append(prediction)

    return predicted_texts

In [ ]:
conversation = prompt_to_conversation(full_prompt)

predicted_text = vllm_generate_choice(model, conversation)

print(50*"-")
print("Input text:")
print(full_prompt)
print()
print()
print("Model's response:")
print(predicted_text[0])
print(50*"-")